# INTRODUCTION
The purpose of this notebook is to explore and understand the data we are working with, clean it and ready it for analysis.  
I am working with the **Online Retail - UCI Machine Learning Repository** dataset.  
It is a transactional data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.  
My goal for the project is to do revenue analysis from actual product sales for different cohorts based only in Europe.

| Variable Name | Role | Type | Description | Units |
|---|---|---|---|---|
| InvoiceNo | ID | Categorical | A 6-digit number uniquely assigned to each transaction. Codes starting with 'C' indicate a cancellation. | — |
| StockCode | ID | Categorical | A 5-digit number uniquely assigned to each distinct product. | — |
| Description | Feature | Categorical | Product (item) name. | — |
| Quantity | Feature | Integer | Number of units of a product per transaction. | — |
| InvoiceDate | Feature | Date | The day and time when the transaction was generated. | — |
| UnitPrice | Feature | Continuous | Price per unit of the product. | Sterling (£) |
| CustomerID | Feature | Categorical | A 5-digit number uniquely assigned to each customer. | — |
| Country | Feature | Categorical | The name of the country where the customer resides. | — |

In [1]:
# Import necessary libraries
import pandas as pd
import warnings
import os

warnings.filterwarnings("ignore")

In [2]:
# Read in the data
data = pd.read_excel(r"../data/raw/Online Retail.xlsx")

european_countries_in_the_data = [
    'United Kingdom', 'France', 'Netherlands', 'Germany', 'Norway',
    'EIRE', 'Switzerland', 'Spain', 'Poland', 'Portugal',
    'Italy', 'Belgium', 'Lithuania', 'Iceland', 'Channel Islands',
    'Denmark', 'Cyprus', 'Sweden', 'Austria', 'Greece', 'Malta',
    'Czech Republic'
]
# filter the data to only include european countries and view first five rows
df = data[data['Country'].isin(european_countries_in_the_data)].copy()
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
# Get the shape of the dataset
print(f"Dataset has {df.shape[0]} rows and {df.shape[1]} columns")

Dataset has 537602 rows and 8 columns


In [4]:
df.describe(include="all")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,537602.0,537602,536148,537602.000000,537602,537602.000000,403061.000000,537602
unique,25678.0,4070,4223,NaN,NaN,NaN,NaN,22
top,573585.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1114.0,2305,2361,NaN,NaN,NaN,NaN,495478
mean,NaN,NaN,NaN,9.361111,2011-07-04 15:46:26.658047488,4.552979,15309.882916,NaN
min,NaN,NaN,NaN,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,2011-03-28 11:34:00,1.250000,13988.000000,NaN
50%,NaN,NaN,NaN,3.000000,2011-07-20 11:54:00,2.080000,15179.000000,NaN
75%,NaN,NaN,NaN,10.000000,2011-10-19 12:35:00,4.130000,16801.000000,NaN
max,NaN,NaN,NaN,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000,NaN


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 537602 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    537602 non-null  object        
 1   StockCode    537602 non-null  object        
 2   Description  536148 non-null  object        
 3   Quantity     537602 non-null  int64         
 4   InvoiceDate  537602 non-null  datetime64[ns]
 5   UnitPrice    537602 non-null  float64       
 6   CustomerID   403061 non-null  float64       
 7   Country      537602 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 36.9+ MB


In [6]:
df.duplicated().sum()

np.int64(5256)

In [7]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     134541
Country             0
dtype: int64

In [8]:
# Get null value percentage per feature
missing_pct = round(100 * (df.isnull().sum()/len(df)) , 2)
missing_pct.sort_values(ascending=False)

CustomerID     25.03
Description     0.27
StockCode       0.00
InvoiceNo       0.00
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
Country         0.00
dtype: float64

**NOTES:**
- Dataset has 537,602 records explained in 8 fields.
- 2 fields have missing values: CustomerID (134,541) and Description (1,454).
- There are 5,256 duplicate records.
- Presence of negative values in Quantity and UnitPrice columns. That's unusual.
- CustomerID column is of float64 datatype. Needs correcting.
- Presence of outlier values in Quantity and UnitPrice columns. Very extreme values and large standard deviation values at a glance.

**To do:**
- Drop duplicate records
- Drop negative Quantity and UnitPrice values
- Drop missing CustomerID records since we want to do cohort and revenue analysis. CustomerID is critical.
- Clip out outliers from the data.
- Standardize description column to lowercase for better aesthetic. Same for column names.
- Check out product stock codes so as to only get actual product sales. Data includes cancellations too.

In [9]:
# Fill NaNs (Incase)  with empty string to avoid errors
stockcodes = df['StockCode'].fillna('')

# Create a new column to classify stock codes
def classify_object(code:str):
    if str(code).isalpha():
        return 'alphabetical'
    elif str(code).isdigit():
        return 'numeric'
    else:
        return 'alphanumeric'

df['stockcode_type'] = stockcodes.apply(classify_object)

# Count rows by type
type_counts = df['stockcode_type'].value_counts()
print("Counts by StockCode type:\n", type_counts)

# Preview examples of each type
for t in ['alphabetical', 'numeric', 'alphanumeric']:
    print(f"\nSample {t} StockCodes:")
    print(df[df['stockcode_type'] == t]['StockCode'].unique()[:20])

Counts by StockCode type:
 stockcode_type
numeric         483161
alphanumeric     51757
alphabetical      2684
Name: count, dtype: int64

Sample alphabetical StockCodes:
['POST' 'D' 'DOT' 'M' 'S' 'AMAZONFEE' 'm' 'DCGSSBOY' 'DCGSSGIRL' 'PADS'
 'B' 'CRUK']

Sample numeric StockCodes:
[71053 22752 21730 22633 22632 84879 22745 22748 22749 22310 84969 22623
 22622 21754 21755 21777 48187 22960 22913 22912]

Sample alphanumeric StockCodes:
['85123A' '84406B' '84029G' '84029E' '82494L' '85099C' '84997B' '84997C'
 '84519A' '85183B' '85071B' '37444A' '37444C' '84971S' '15056BL' '15056N'
 '35004C' '85049A' '85099B' '84970S']


Some entries in the `StockCode` column do not represent actual product sales.
These are operational, administrative, or accounting entries and should be excluded from revenue and cohort analysis.

| StockCode | Meaning | Category |
|---|---|---|
| `POST` | Postage charges billed to customers | Logistics |
| `DOT` | Dotcom postage | Logistics |
| `PADS` | inspect descriptions| Investigate |
| `M` | Manual adjustment — ad hoc corrections posted by staff | Adjustment |
| `m` | Lowercase variant of `M` — same meaning, likely a data entry inconsistency | Adjustment |
| `D` | Discount — posted as a separate negative line rather than adjusted on the original invoice | Adjustment |
| `B` | Bad debt write-off — unpaid invoices written off; the original sale already occurred | Adjustment |
| `S` | Sample — goods sent to customers free of charge | Adjustment |
| `AMAZONFEE` | Commission deducted by Amazon for channel sales | Fee |
| `CRUK` | Charitable commission passed to Cancer Research UK | Fee |
 `DCGSSBOY` | inspect descriptions | Investigate |
| `DCGSSGIRL` | inspect descriptions | Investigate |

In [10]:
df[df['StockCode'].isin (['PADS', 'DCGSSBOY', 'DCGSSGIRL'])]['Description'].unique()

array(['BOYS PARTY BAG', 'GIRLS PARTY BAG', 'PADS TO MATCH ALL CUSHIONS'],
      dtype=object)

## Cleaning The Data

In [11]:
# Standardize column names
df.rename(columns={
    'InvoiceNo': 'invoice_no',
    'StockCode': 'stock_code',
    'InvoiceDate': 'invoice_date',
    'UnitPrice': 'unit_price',
    'CustomerID': 'customer_id'
}, inplace=True)
df.columns = df.columns.str.lower()

In [12]:
# Remove duplicates
df.drop_duplicates(inplace=True)

In [13]:
# Clean the description field
df['description'] = df['description'].str.strip() # remove spaces
df['description'] = df['description'].str.lower() # convert to lowercase  
df['description'] = df['description'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)  # remove possible weird characters

In [14]:
# Drop cancellations first
df = df[~df['invoice_no'].astype(str).str.startswith('C')]

In [15]:
# Remove negative values
df = df[df['quantity'] > 0] 
df = df[df['unit_price'] > 0]

In [16]:
# Keep only real sales (not returns, nor adjustments) 
# Keeps PADS, DCGSSBOY and DCGSSGIRL. They are actual products.
irrelevant_codes = ['POST', 'D', 'DOT', 'M', 'S', 'AMAZONFEE', 'm', 'B', 'CRUK'] 

# Convert invoice_no to string
df['invoice_no'] = df['invoice_no'].astype(str)

# Keep only sales of relevant stock products
sales = df[~df['stock_code'].isin(irrelevant_codes)]
sales.drop(columns=["stockcode_type"], inplace=True)
sales.reset_index(drop=True, inplace=True)

In [17]:
# Drop Rows with No CustomerID
sales= sales[sales['customer_id'].notna()]

In [18]:
# Correct column data type
sales['customer_id'] = sales['customer_id'].astype(int)
sales['invoice_no'] = sales['invoice_no'].astype(int)

In [19]:
# Drop Outlier values that may distort analysis
sales = sales[
    (sales['quantity'] <= sales['quantity'].quantile(0.99)) &
    (sales['unit_price'] <= sales['unit_price'].quantile(0.99)) &
    (sales['unit_price'] >= sales['unit_price'].quantile(0.01))
]

### Additional Cleaning Step
After checking, I found that there were products with different descriptions yet having similar stock codes. So I had to come back to cleaning and normalize the product descriptions.   
My choice: Use the most prevalent description

In [20]:
sales['description'] = sales.groupby('stock_code')['description'].transform(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])

## Sanity Check

In [21]:
sales.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,white hanging heart tlight holder,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,white metal lantern,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,cream cupid hearts coat hanger,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,knitted union flag hot water bottle,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,red woolly hottie white heart,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [22]:
len(sales[sales['stock_code'].isin(irrelevant_codes)] & sales['invoice_no'].astype(str).str.startswith('C'))

0

In [23]:
sales.describe()

,invoice_no,quantity,invoice_date,unit_price,customer_id
count,377830.000000,377830.000000,377830,377830.000000,377830.000000
mean,560604.687314,9.763274,2011-07-10 21:02:02.056639488,2.713975,15317.042622
min,536365.000000,1.000000,2010-12-01 08:26:00,0.210000,12347.000000
25%,549189.000000,2.000000,2011-04-07 10:11:00,1.250000,13999.000000
50%,561894.000000,6.000000,2011-07-31 15:00:00,1.950000,15194.000000
75%,572092.000000,12.000000,2011-10-20 14:41:00,3.750000,16809.000000
max,581587.000000,120.000000,2011-12-09 12:50:00,12.750000,18287.000000
std,13112.021107,14.187207,NaN,2.463400,1699.776879


In [24]:
sales.isnull().sum()

invoice_no      0
stock_code      0
description     0
quantity        0
invoice_date    0
unit_price      0
customer_id     0
country         0
dtype: int64

In [25]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 377830 entries, 0 to 518737
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   invoice_no    377830 non-null  int64         
 1   stock_code    377830 non-null  object        
 2   description   377830 non-null  object        
 3   quantity      377830 non-null  int64         
 4   invoice_date  377830 non-null  datetime64[ns]
 5   unit_price    377830 non-null  float64       
 6   customer_id   377830 non-null  int64         
 7   country       377830 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(3), object(3)
memory usage: 34.0+ MB


## Add a Total Price Feature

In [26]:
sales['total_price'] = round(sales['quantity'] * sales['unit_price'], 2)

In [30]:
sales.shape

(377830, 9)

## Export data to csv

In [29]:
output_dir = '../data/cleaned'
os.makedirs(output_dir, exist_ok=True)

sales.to_csv(os.path.join(output_dir, 'SalesData.csv'), index=False)